In [ ]:
import logging

import sys
import os
# Add the parent directory (root of the project) to the path
sys.path.append(os.path.abspath('..'))
import shared


import polars as pl
import numpy as np


import seaborn as sns
import matplotlib.pyplot as plt

from feature_engineering import build_features

In [ ]:
LOG_FMT = (
    "%(asctime)s - %(name)s [%(threadName)s] %(funcName)s [%(levelname)s] %(message)s"
)
logging.basicConfig(level=logging.INFO, format=LOG_FMT)

In [ ]:
os.environ['RACE_TYPE'] = "ve"
os.environ['FORECAST_YEAR'] = "2025"
runs_path = f"../data/long_runs_and_running_order_{shared.race_id_str()}.tsv"

logging.info(f"Reading {runs_path}")
runs_df = pl.read_csv(runs_path, separator="	")
runs_df

In [ ]:
forecast_year = shared.forecast_year()
bc_df, feature_names, boxcox_params = build_features(runs_df, forecast_year)

logging.info(
    f"{shared.race_id_str()} {boxcox_params.lmbda=}, {boxcox_params.bc_mean=}, {boxcox_params.bc_std=}"
)

In [ ]:
example_name = "oskari pirttikoski"
example_team = "REAKTOR"

if shared.race_type() == "ve":
    example_name = "mari sane"
    example_team = "VIILEÄT VENLAT"

if shared.race_type() == "ke":
    example_name = "anna-liisa käppi"
    example_team = "KENRAALI"

In [ ]:
history_df = bc_df.filter(
    (
        pl.col("year") >= 2004  # Quick hack to Reduce num of no history
    )
).with_columns(pl.col("pace").clip(upper_bound=25).alias("capped_pace"))
history_df.glimpse()

In [ ]:
history_df.select("team_country").describe()

#history_df.with_columns(
#    pl.col('team_country').is_null().alias("null_country")
#).group_by(["year", "null_country"]).count().sort(by=["null_country", "count"], descending=True)
#history_df.columns
#history_df.describe()

In [ ]:
# assert history_df['team_country'].null_count() == 0
required = ['run_id',
 'linked_runner_id',
 'name',
 'team_id',
 'team',
 'team_country',
 'year',
 'leg',
 'run_num',
 "num_runs", 
 "leg_dist",
    "marking_norm", "vertical_coef", "normalized_team_id", "normalized_leg_dist", "run_num_norm", "first_time"            
]

null_counts = history_df.select(pl.col(required).null_count())
bad = {c: null_counts[c][0] for c in required if null_counts[c][0] > 0}
assert not bad, f"Columns with nulls: {bad}"


In [ ]:
history_df.select("capped_pace").describe()

In [ ]:
history_df.select(feature_names).describe()

In [ ]:
feature_df = history_df.select(feature_names).to_pandas()
corr_matrix = feature_df.corr(numeric_only=True)

abs_corr = corr_matrix.abs().copy()
np.fill_diagonal(abs_corr.values, 0.0)

strongest_pairs = (
    abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
    .head(10)
    .rename("abs_corr")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)

top_feature_names = sorted(
    set(strongest_pairs["feature_a"]).union(strongest_pairs["feature_b"])
)

top_corr_matrix = corr_matrix.loc[top_feature_names, top_feature_names]

plt.figure(figsize=(12, 9))
sns.heatmap(
    top_corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
plt.title("Features appearing in the top 10 correlated pairs")
plt.tight_layout()

display(strongest_pairs)
# display(top_corr_matrix)

In [ ]:
race_type = shared.race_type()


for column_name in [
    "normalized_team_id",
    "c_bcp_median",
]:
    # logging.info(f'{column_name} stats:\n{bc_df[column_name].describe()}')
    plt.figure(figsize=(10, 6))

    ax = sns.kdeplot(history_df, x=column_name, palette="bright")
    ax.set_title(f"{race_type.upper()} {column_name}")

In [ ]:
for column_name in feature_names:
    # logging.info(f'{column_name} stats:\n{bc_df[column_name].describe()}')
    plt.figure(figsize=(10, 6))

    ax = sns.kdeplot(history_df, x=column_name, hue="year", palette="bright")
    ax.set_title(f"{race_type.upper()} {column_name}")

In [ ]:
for column_name in feature_names:
    # logging.info(f'{column_name} stats:\n{bc_df[column_name].describe()}')
    plt.figure(figsize=(10, 6))

    # ax = sns.lineplot(history_df,x=column_name, y='pace', hue='first_time',  palette='bright')
    ax = sns.lmplot(
        history_df,
        x=column_name,
        y="capped_pace",
        hue="first_time",
        height=8,
        aspect=1.7,
        ci=50,
        scatter_kws={"alpha": 0.07},
        x_jitter=0.0004,
    )
    # ax.set_title(f"{race_type.upper()} {column_name}")